In [ ]:
from pathlib import Path
import sys
import joblib
import numpy as np
import pandas as pd
from pathlib import Path
from feature_analysis import find_high_correlation_pairs
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
    roc_curve,
 )

sys.path.insert(0, str(Path("..").resolve()))
from backend.registry import register_dataset, register_model

MODELS_DIR = Path("models")
MODELS_DIR.mkdir(exist_ok=True)
sys.path.insert(0, str(Path("..").resolve()))

In [ ]:
df = pd.read_csv('brfss_2024.zip', compression='zip')
pd.set_option('display.max_columns', None)

target_col = 'CVDSTRK3'

# Keep only rows where we have a clear stroke status
df = df[(df[target_col] == 1) | (df[target_col] == 2)]

# Recode BRFSS survey values (1=Yes, 2=No) to standard binary (1=Stroke, 0=No Stroke)
df[target_col] = (df[target_col] == 1).astype(int)

print("Amount of rows and features:", df.shape)
df.head(1)

In [ ]:
# Manual feature selection - the goal is to drop features such as administrative and survey design variables.
# These variables tell how the data was collected, not what respondents reported about their health or behaviour.

df.drop(columns=[
    # When the survey was conducted and general identifiers whether is eligible to be included in the survey
    "FMONTH", "IDATE", "IMONTH", "IDAY", "IYEAR", "DISPCODE", "SEQNO", "_PSU", "SAFETIME",
    "LADULT1", "CADULT1", "RESPSLC1", "CTELENM1", "CTELNUM1","CELPHON1", "CELLFON5", "LANDLINE",
    "LANDSEX3", "CELLSEX3",
    
    # Residency and household variables
    "PVTRESD1", "PVTRESD3", "COLGHOUS", "STATERE1", "CSTATE1", "NUMADULT", 
    "HHADULT", "NUMHHOL4", "NUMPHON4", "CPDEMO1C", "CCLGHOUS", "GUNLOAD", 
    "LOADULK2", "FIREARM5", "_CHLDCNT", "CHILDREN", "CAGEG", "RENTHOM1",

    # Survey language and version variables
    "QSTVER", "QSTLANG", "ICFQSTVR",

    # Statistical sampling weights & design variables
    "_STSTR", "_STRWT", "_RAWRAKE", "_WT2RAKE", "_CLLCPWT", 
    "_DUALUSE", "_DUALCOR", "_LLCPWT2", "_LLCPWT",

    # Post Course of Treatment questions
    "CSRVINST", "CSRVINSR", "CSRVDEIN", "PCSTALK2", "CSRVSUM",

    # Caregiver - about the respondent's relationship to their caregiver, 
    # not about the respondent's health or behaviour.
    "CAREGIV1", "CRGVREL5", "CRGVPRB4", "CRGVALZD", 
    "CRGVNURS", "CRGVPER2", "CRGVHOU2", "CRGVHRS2", "CRGVLNG2", 

    # Family Planning
    "RCSRLTN2", "HADSEX", "PFPPRVN4", "TYPCNTR9", "NOBCUSE8",

    "RCSXBRTH", "RCSGEND1", "HPVDSHT", "CSRVCTL2",
], inplace=True)

print("Dataset after dropping columns that are unrelated to health outcomes by design:", df.shape)
df.head(1)

In [ ]:
# Find highly correlated features using Cramér's V for categorical variables.

# pairs_df = find_high_correlation_pairs(df, threshold=0.85, sample_n = 50_000, random_state = 42)
# print(pairs_df.to_string(index=False))

In [ ]:
# Remove redundant features

df.drop(columns=[
    # General health / days unhealthy
    # Keep: GENHLTH, PHYSHLTH, MENTHLTH, _TOTINDA 
    # Precise amount of days unhealthy in the past 30 days, while the alternatives are coarser categories or binary variables
    "_RFHLTH", "_PHYS14D", "_MENT14D", "POORHLTH", "EXERANY2",

    # BMI / anthropometrics
    # Keep: _BMI5 (continuous computed BMI)
    # Raw height and weight in all unit encodings are redundant as we have _BMI5. 
    # _BMI5CAT and _RFBMI5 are redundant to _BMI5.
    "WEIGHT2", "HEIGHT3", "HTIN4", "HTM4", 
    "WTKG3", "_BMI5CAT", "_RFBMI5",

    # Race / ethnicity
    # Keep: _RACE (8-category combined variable covering all race variants)
    "_RACEG21", "_RACEGR3", "_RACEPRV", "_MRACE1", 
    "_IMPRACE", "_CHISPNC", "_CRACE1", "_HISPANC",

    # Age
    # Keep: _AGE80 (Alternatives are encoded in age groups, but _AGE80 is a continuous number 
    # that gives us more precise information about the respondent's age)
    "_AGE65YR", "_AGE_G", "_AGEG5YR", "_LCSAGE",

    # Smoking
    # Keep: _SMOKER3 (Derived variable that combines current / former / never / every day smoking status)
    # Keep: LCSNUMC_ (preciese number of cigarettes smoked per day)
    # Raw inputs SMOKE100 and SMOKDAY2 are fully absorbed by _SMOKER3.
    # _RFSMOK3 is a binary collapse of _SMOKER3. USENOW3, LASTSMK2,
    "_RFSMOK3", "USENOW3", "LASTSMK2", "STOPSMK2", "LCSNUMCG",
    "HEATTBCO", "SMOKE100", "SMOKDAY2", "_PACKDAY", "_LCSSMKG",
    "_PACKYRS",

    # E-cigarettes / vaping
    # Keep: ECIGNOW3 (Has all 3-category current / former / never status)
    "_CURECI3", "MENTCIGS", "MENTECIG",
    
    # Marijuana / cannabis
    # Keep: MARIJAN1 (Give us not just the status but precise amount of consumption days)
    "MARJEAT", "MARJDAB", "MARJOTHR", "MARJSMOK", "MARJVAPE", "USEMRJN4",

    # Alcohol
    # Keep: _DRNKWK3 (Calculated total number of alcoholic beverages consumed per week, 
    # while the alternatives are coarser categories or binary variables)
    "_RFBING6", "DRNKANY6", "ALCDAY4", "DRNK3GE5", 
    "MAXDRNKS", "DROCDY4_", "_RFDRHV9", "AVEDRNK4",

    # Education / Income
    # Keep: EDUCA / INCOME3 (EDUCA and INCOME3 respectively provide a more detailed range of education and income levels)
    "_EDUCAG", "_INCOMG1", 

    # Health insurance / access
    # Keep: PRIMINS2 , PERSDOC3, CHECKUP1
    "MEDCOST1", "_HLTHPL2" , "_HCVU654",

    # Sex
    # Keep: _SEX
    "SEXVAR",

    # Asthma
    # Keep: _ASTHMS1 (Computed Asthma Status, covers all 3 current, former and never asthma status)
    "_LTASTH1", "_CASTHM1", "CASTHDX2", "CASTHNO2", "ASTHNOW", "ASTHMA3", 

    # Cancer / Lung cancer screening test
    # Keep: _LCSCTSN, CERVSCRN, 
    "CSRVDOC1", "CSRVRTRN", "CSRVCLIN", "LCSSCNCR", "LCSCTSC1", "_LCSPSTF",
    "LCSCTWHN", 

    # Dental / oral health
    # Keep: LASTDEN4, RMVTETH4
    "_EXTETH3", "_DENVST3", "_ALTETH3",

    # PSA test follow-ups
    # Keep: PSATEST1 (ever had PSA test)
    "PSATIME1", "PCPSARS2", "PSASUGS1",

    # Mammography / cervical screening
    # Keep: HOWLONG
    "_SGMS102", "_SGMSCP2", "_CLNSCP2", "_MAM402Y", "_RFMAM23",
    "_CRVSCRN", "_RFPAP37",

    # Flu shot
    # FLUSHOT7 (as it has no age restriction, while _FLSHOT7 is only for 65+)
    "_FLSHOT7", "_PNEUMO3", "FLSHTMY3", "IMFVPLA5", "HIVTSTD3",

    # Colorectal screening
    # Keep: COLNSIGM (handles both sigmoidoscopy and colonoscopy, as well as both)
    "_HADSIGM", "_HADCOLN", "VIRCOLO1", "_VIRCOL2", "_RFBLDS6", "_STOLDN2",
    "STOOLDN2", "_CRCREC3", "BLDSTFIT", "SDNATES1", "VCLNTES2", "SMALSTOL",
    "_SBONTI2", "HADSIGM4", "LASTSIG4",
    
    # Arthritis
    # Keep: HAVARTH4
    "_DRDXAR2", "ARTHEXER", "CHKHEMO3", 

    # Urbanicity
    # Keep: _URBSTAT
    "_METSTAT", "MSCODE",

    # HIV/AIDS
    # Keep: HIVTST7
    "_AIDTST4", "_HPV5YR1", "_PAPHPV1", "HPVADSH1",

    # Angina or Coronary Heart Disease
    # Keep: _MICHD and CVDINFR4
    "CVDCRHD4",
], inplace=True)

print("Dataset after dropping redundant features:", df.shape)
df.head(1)

In [ ]:
# Compare the results of correlation analysis after removing redundant features
# from feature_analysis import find_high_correlation_pairs

# pairs_df = find_high_correlation_pairs(df, threshold=0.80, sample_n = 50_000, random_state = 42)
# print(pairs_df.to_string(index=False))

In [ ]:
# Features categorised per the Overleaf report tables
# Table 1 (tab:stroke_feature_rationale) — included features
included_features = [
    # Demographic
    '_STATE', '_SEX', '_RACE', '_AGE80',
    'EDUCA', 'INCOME3', 'MARITAL', 'EMPLOY1', 'VETERAN3', 'PREGNANT',
    '_URBSTAT',

    # General health
    'GENHLTH', 'PHYSHLTH', 'MENTHLTH',

    # Lifestyle / behavioural
    '_TOTINDA', '_BMI5',
    '_SMOKER3', 'LCSFIRST', 'LCSLAST_', 'LCSNUMC_', '_LCSYSMK', '_LCSYQTS',
    '_LCSELIG', '_LCSCTSN',
    'ECIGNOW3', 'MARIJAN1',
    '_DRNKWK3',
    'SSBSUGR2', 'SSBFRUT3',

    # Diabetes
    'DIABETE4', 'DIABAGE4', 'DIABTYPE', 'PDIABTS1', 'PREDIAB2',
    'INSULIN1', 'EYEEXAM1', 'DIABEYE1', 'DIABEDU1', 'FEETSORE',

    # Clinical comorbidities
    '_ASTHMS1', 'CHCCOPD3', 'CHCKDNY2', 'HAVARTH4', 'ADDEPEV3',
    'CHCOCNC1', 'CNCRTYP2', 'CSRVTRT3', 'CNCRAGE', 'CSRVPAIN',

    # Healthcare access
    'PRIMINS2', 'PERSDOC3', 'LASTDEN4', 'RMVTETH4',

    # Female-specific
    'HADHYST2',

    # Adverse childhood experiences (ACEs)
    'ACEDEPRS', 'ACEDRINK', 'ACEDRUGS', 'ACEPRISN', 'ACEDIVRC',
    'ACEPUNCH', 'ACEHURT1', 'ACESWEAR',
    'ACETOUCH', 'ACETTHEM', 'ACEHVSEX',
    'ACEADSAF', 'ACEADNED',

    # Social / psychosocial
    'LSATISFY', 'EMTSUPRT', 'SDLONELY',
]

# Table 2 (tab:stroke_feature_exclusion_rationale) — excluded features
excluded_features = [
    # Healthcare engagement proxies with no independent stroke signal
    'CHECKUP1',
    'HADMAM', 'HOWLONG',
    'CERVSCRN', 'CRVCLCNC', 'CRVCLPAP', 'CRVCLHPV',
    'COLNSIGM', 'COLNTES1', 'SIGMTES1', 'COLNCNCR', 'STOLTEST',
    'HIVTST7', 'HIVRISK5',
    'PSATEST1',
    # Vaccines — no causal stroke risk
    'SHINGLE2', 'HPVADVC4', 'TETANUS1',
    # Social-needs / material-hardship — redundant with income/education
    'FOODSTMP', 'SDHFOOD1', 'SDHBILLS', 'SDHUTILS', 'SDHTRNSP',
    # Weak / non-specific cancer
    'CHCSCNC1',
]

df.drop(columns=excluded_features, inplace=True)
print("Dataset after dropping excluded features:", df.shape)

# Table 3 (tab:stroke_feature_uncertainty_rationale) — uncertain features
uncertain_features = [
    # Post-stroke functional limitations (temporal ambiguity)
    'DIFFWALK', 'DIFFDRES', 'DIFFALON',
    # Cognitive items (pre- vs post-stroke unclear)
    'CIMEMLO1', 'CDWORRY', 'CDDISCU1', 'CDHOUS1', 'CDSOCIA1', 'DECIDE',
    # Social-needs items with directional ambiguity
    'SDHEMPLY',
    # Vaccinations with uncertain directionality
    'FLUSHOT7', 'PNEUVAC4',
    # Sensory / neighbourhood items
    'DEAF', 'BLIND', 'HOWSAFE1',
    # Sexual orientation
    'SOMALE', 'SOFEMALE',
    # Cancer count (likely redundant)
    'CNCRDIFF',
    # CVD variables (overlap between _MICHD, CVDINFR4, CVDCRHD4)
    '_MICHD', 'CVDINFR4', 'CVDCRHD4',
]

In [ ]:
for col in df.columns:
    print(f"{col}: {df[col].unique()}")

In [ ]:
# BRFSS 2024 encodes "Don't know / Refused / Missing" responses with specific codes that vary by question.
# The problem is that a high value for those responses can be misinterpreted by models as meaningful data (have a high weight on the model).
# The code below replaces those specific codes with NaN, and in some cases 0 (for "none" or "never" responses).

# Handle don't know / unsure / refused / none / never

# 7/9 → NaN (don't know / refused)
_cols_dk = [
    "GENHLTH", "PERSDOC3", "CVDINFR4", "CHCOCNC1", "CHCCOPD3",
    "ADDEPEV3", "CHCKDNY2", "HAVARTH4", "DIABETE4", "VETERAN3",
    "PREGNANT", "DEAF", "BLIND", "DECIDE", "DIFFWALK", "DIFFDRES", "DIFFALON",
    "HADHYST2", "ECIGNOW3", "FLUSHOT7", "PNEUVAC4", "PREDIAB2",
    "DIABTYPE", "INSULIN1", "FEETSORE", "CNCRDIFF", "CSRVTRT3", "CSRVPAIN", 
    "CIMEMLO1", "CDWORRY", "CDDISCU1", "CDHOUS1", "CDSOCIA1", "ACEDEPRS", "ACEDRINK", "ACEDRUGS",
    "ACEPRISN", "ACEPUNCH", "ACEHURT1", "ACESWEAR", "ACETOUCH", "ACETTHEM",
    "ACEHVSEX", "ACEADSAF", "ACEADNED", "LSATISFY", "EMTSUPRT", "SDLONELY",
    "SDHEMPLY", "HOWSAFE1", "SOMALE", "SOFEMALE"
]

for col in _cols_dk:
    df[col] = df[col].replace({7: np.nan, 9: np.nan})

# 7/9 → NaN, 8 → 0 (none/never)
_cols_dk_none = [
    "LASTDEN4", "RMVTETH4", "DIABEYE1", "DIABEDU1", "PDIABTS1",
]
for col in _cols_dk_none:
    df[col] = df[col].replace({7: np.nan, 8: 0, 9: np.nan})

# 77/88/99 → NaN/0/NaN
for col in ["PRIMINS2", "PHYSHLTH", "MENTHLTH", "MARIJAN1"]:
    df[col] = df[col].replace({77: np.nan, 88: 0, 99: np.nan})

# 77/99 → NaN (no 88 code)
for col in ["CNCRTYP2", "INCOME3"]:
    df[col] = df[col].replace({77: np.nan, 99: np.nan})

# 777/888/999 → NaN/0/NaN
for col in ["SSBSUGR2", "SSBFRUT3", "LCSFIRST"]:
    df[col] = df[col].replace({777: np.nan, 888: 0, 999: np.nan})

# 98/99 → NaN (age fields)
for col in ["DIABAGE4", "CNCRAGE"]:
    df[col] = df[col].replace({98: np.nan, 99: np.nan})

# 9 → NaN (where 9 stands for Don't know / Refused / Missing)
for col in ["_TOTINDA", "_ASTHMS1", "_RACE", "_SMOKER3", "EDUCA", "MARITAL"]:
    df[col] = df[col].replace({9: np.nan})

# Irregular codings — handled individually
df["ACEDIVRC"] = df["ACEDIVRC"].replace({7: np.nan, 8: 3, 9: np.nan})
df["EYEEXAM1"] = df["EYEEXAM1"].replace({6: np.nan, 7: np.nan, 8: 0, 9: np.nan})
df["_DRNKWK3"] = df["_DRNKWK3"].replace({99900: np.nan})

# BMI and alcohol consumption are stored times 100 (e.g. 2249 = BMI 22.49, 1000 = 10.00 drinks per week)
# Scale them to the normal range to avoid misleading
df['_BMI5'] = df['_BMI5'] / 100
df['_DRNKWK3'] = df['_DRNKWK3'] / 100

# For smoking duration, negative values are likely data errors
df['_LCSYQTS'] = df['_LCSYQTS'].where(df['_LCSYQTS'] >= 0, other=pd.NA)

for col in ['_DRNKWK3', '_LCSYSMK', '_LCSYQTS']:
    df[col] = df[col].round(0).astype('Int64')

In [ ]:
for col in df.columns:
    print(f"{col}: {df[col].unique()}")

In [ ]:
combined_cols = [column for column in df.columns if column != target_col]

# Shared cols (both for lifestyle and clinical)
shared_cols = ['_SEX']

# Lifestyle-only cols
lifestyle_cols = ['_TOTINDA']

clinical_only_cols = [
    c for c in combined_cols
    if c not in set(lifestyle_cols + shared_cols)
]

# Two variants: with and without uncertain features
uncertain_set = set(uncertain_features)

def drop_uncertain(cols):
    """Return cols with uncertain features removed."""
    return [c for c in cols if c not in uncertain_set]

dataset_definitions = {
    'with_uncertain': {
        'lifestyle': lifestyle_cols + shared_cols,
        'clinical': clinical_only_cols + shared_cols,
        'combined': combined_cols,
    },
    'without_uncertain': {
        'lifestyle': drop_uncertain(lifestyle_cols) + drop_uncertain(shared_cols),
        'clinical': drop_uncertain(clinical_only_cols) + drop_uncertain(shared_cols),
        'combined': drop_uncertain(combined_cols),
    },
}

# Register datasets
datasets = {}

for uncertainty_variant, feature_sets in dataset_definitions.items():
    for feat_name, feat_cols in feature_sets.items():
        # Only keep cols that actually exist in df
        valid_cols = [c for c in feat_cols if c in df.columns]
        ds = df[valid_cols + [target_col]]

        dataset_id = f"{feat_name}_{uncertainty_variant}"
        datasets[dataset_id] = {
            'df': ds,
            'feature_set': feat_name,
            'uncertainty': uncertainty_variant,
        }
        register_dataset(dataset_id, ds, label=dataset_id.replace('_', ' ').title())
        print(f"Registered dataset: {dataset_id}  features={ds.shape[1] - 1}")

print(f"\nTotal datasets: {len(datasets)}")

# Algorithms
algorithms = {
    'random_forest': RandomForestClassifier(n_estimators=100, random_state=42),
    'xgboost': XGBClassifier(eval_metric='logloss', random_state=42),
    'lightgbm': LGBMClassifier(random_state=42, verbose=-1),
}

# Train: 3 algorithms * 6 datasets = 18 models
for dataset_id, dataset_info in datasets.items():
    ds = dataset_info['df']
    feat_name = dataset_info['feature_set']
    uncertainty = dataset_info['uncertainty']

    X, y = ds.drop(columns=[target_col]), ds[target_col]
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42
    )

    for algo_name, clf in algorithms.items():
        model_id = f"{algo_name}_{dataset_id}"
        print(f"Training {model_id}...")

        clf.fit(X_train, y_train)
        y_pred = clf.predict(X_test)
        y_prob = clf.predict_proba(X_test)[:, 1]

        pkl_path = (MODELS_DIR / f"{model_id}.pkl").resolve()
        joblib.dump(clf, pkl_path)

        report_raw = classification_report(y_test, y_pred, output_dict=True)
        cm = confusion_matrix(y_test, y_pred)
        fpr, tpr, _ = roc_curve(y_test, y_prob)
        importances = getattr(clf, 'feature_importances_', np.zeros(len(X.columns)))

        register_model(
            model_id=model_id,
            algorithm=algo_name,
            feature_set=feat_name,
            uncertainty_variant=uncertainty,
            model_path=str(pkl_path),
            metrics={
                'auc': roc_auc_score(y_test, y_prob),
                'accuracy': accuracy_score(y_test, y_pred),
                'f1': f1_score(y_test, y_pred),
                'precision': precision_score(y_test, y_pred),
                'recall': recall_score(y_test, y_pred),
            },
            classification_report={
                'classes': {
                    k: v for k, v in report_raw.items()
                    if k not in ('accuracy', 'macro avg', 'weighted avg')
                },
                'macro_avg': report_raw['macro avg'],
                'weighted_avg': report_raw['weighted avg'],
                'accuracy': report_raw['accuracy'],
            },
            confusion_matrix={
                'tn': int(cm[0, 0]), 'fp': int(cm[0, 1]),
                'fn': int(cm[1, 0]), 'tp': int(cm[1, 1]),
            },
            feature_importances=[
                {'feature': col, 'importance': float(imp)}
                for col, imp in zip(X.columns, importances)
            ],
            roc_curve={'fpr': fpr.tolist(), 'tpr': tpr.tolist()},
            feature_columns=list(X.columns),
        )
        print(f"Registered {model_id}")

print(f"\nDone. {len(datasets) * len(algorithms)} models trained and registered.")